<p align="center">
  <a href="https://colab.research.google.com/github/AiRA-Laboratory/AI4SE/blob/main/module_1_nn_cnn/TA_NguyenDinhHai/regression_clean.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" height="40px">
  </a>
</p>

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# 1. Tạo dữ liệu gốc y = 2x + 1 dính nhiễu lớn
X = np.linspace(-2, 2, 300).reshape(-1, 1)
y = 2 * X + 1 + np.random.normal(0, 0.4, X.shape)

# Xáo trộn dữ liệu để tập Học và tập Thi đồng đều nhau
indices = np.arange(len(X))
np.random.shuffle(indices)
X, y = X[indices], y[indices]

# 2. Chuẩn hóa Min-Max dữ liệu về đoạn [0, 1]
X_norm = (X - X.min()) / (X.max() - X.min())
y_norm = (y - y.min()) / (y.max() - y.min())

# TODO 1: Chuyển đổi dữ liệu từ mảng NumPy sang PyTorch Tensor kiểu Float
# Gợi ý: Sử dụng hàm torch.FloatTensor(...) cho X_norm và y_norm
X_tensor = ?
y_tensor = ?

# 3. Chia tập dữ liệu thành 2 phần: Học (Train - 80%) và Thi (Test - 20%)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [ ]:
# Mạng nơ-ron hồi quy tuyến tính sâu phục vụ thử nghiệm Dropout
model = nn.Sequential(
    nn.Linear(1, 32),
    # TODO 2: Khai báo lớp Dropout để ngẫu nhiên tắt bớt nơ-ron chống học vẹt (Overfitting)
    # Gợi ý: Sử dụng lớp nn.Dropout(...) với tỷ lệ tắt nơ-ron là 0.1 (10%)
    ?,
    nn.Linear(32, 1)
)

In [ ]:
# TODO 3: Khai báo thước đo sai số (Hàm Loss) phù hợp cho bài toán Hồi quy (Regression)
# Gợi ý: Dùng hàm tính Bình phương sai số trung bình (Mean Squared Error) của PyTorch
criterion = ?

# TODO 4: Khai báo thuật toán tối ưu để vặn các nút trọng số và sửa sai lỗi của mạng
# Gợi ý: Sử dụng lớp optim.Adam(...) với tốc độ học lr = 0.02, nhớ truyền model.parameters() vào
optimizer = ?

train_losses, test_losses = [], []
patience, patience_counter = 10, 0
best_test_loss = float('inf')
max_epochs = 500

In [ ]:
for epoch in range(max_epochs):
    # --- GIAI ĐOẠN 1: HỌC (TRAINING) ---
    # TODO 5: Kích hoạt chế độ huấn luyện cho mô hình để lớp Dropout bắt đầu hoạt động
    # Gợi ý: Gọi hàm .train() của đối tượng model
    ?

    optimizer.zero_grad()

    # TODO 6: Thực hiện lan truyền tiến (Forward Pass) để máy đưa ra dự đoán đầu ra
    # Gợi ý: Cho tập dữ liệu X_train chạy đi qua mạng nơ-ron "model"
    y_pred_train = ?
    train_loss = criterion(y_pred_train, y_train)

    # TODO 7: Thực hiện lan truyền ngược (Backward Pass) để tính toán ma trạng đạo hàm lỗi
    # Gợi ý: Gọi hàm .backward() xuất phát từ biến lưu sai số train_loss
    ?
    optimizer.step()

    # --- GIAI ĐOẠN 2: THI THỬ (TESTING) ---
    # TODO 8: Kích hoạt chế độ đánh giá để tạm thời tắt lớp Dropout khi đi kiểm tra tập Test
    # Gợi ý: Gọi hàm .eval() của đối tượng model
    ?

    with torch.no_grad():
        test_loss = criterion(model(X_test), y_test)

    train_losses.append(train_loss.item())
    test_losses.append(test_loss.item())

    # --- Logic kiểm tra Early Stopping ---
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early Stopping activated at Epoch {epoch + 1}")
        break

In [ ]:
# Tải lại mô hình ở thời điểm đỉnh cao nhất trước khi bị học vẹt
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

with torch.no_grad():
    predictions = model(X_tensor).numpy()

# Tiến hành vẽ đồ thị phân tích kết quả
plt.figure(figsize=(12, 4))

# Đồ thị 1: So sánh Lỗi tập Học và Lỗi tập Thi để giải thích Early Stopping
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', color='blue')
plt.plot(test_losses, label='Test Error', color='orange')
plt.title('Đồ thị so sánh lỗi: Học và Thi')
plt.xlabel('Vòng lặp (Epochs)')
plt.ylabel('Giá trị lỗi (MSE)')
plt.legend()

# Đồ thị 2: Đường hồi quy màu đỏ đi xuyên qua đống dữ liệu thực tế
plt.subplot(1, 2, 2)
plt.scatter(X_norm, y_norm, s=10, color='gray', alpha=0.5, label='Dữ liệu thực tế')
plt.plot(X_norm, predictions, color='red', linewidth=2, label='Đường thẳng máy đoán')
plt.title('Kết quả đường hồi quy thực tế')
plt.xlabel('X (Chuẩn hóa)')
plt.ylabel('y (Chuẩn hóa)')
plt.legend()

plt.tight_layout()
plt.show()